# LLM Fiscal Policy Classification Demo

Demonstrates fiscal and monetary policy classification prompts with 4 variants each: simple, with_definitions, chain_of_thought, and few_shot.

In [1]:
# Setup
from dotenv import load_dotenv
load_dotenv('../../.env')
import os, sys
sys.path.insert(0, '../../libs')
sys.path.insert(0, '../../src/Traction/prompts')
sys.path.insert(0, '../../src/Traction')
import config
from llm_factory_openai import LLMAgent
# Use _process_batch_async to process batch_messages
import asyncio
from llm_batch_process_utils import _process_batch_async
from llm_factory_openai import BatchAsyncLLMAgent
from llm_batch_process_utils import _merge_ids_with_responses

from prompt_utils import load_prompt, format_messages
from schema import PROMPT_REGISTRY
from pathlib import Path
import pandas as pd
from llm_batch_process_utils import _build_batch_messages_from_df_flexible
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix, precision_score, recall_score

#### Set up llm

In [4]:
# Initialize LLM Agent
llm_agent = LLMAgent(
    api_key=os.getenv('OPENAI_API_KEY'),
    model='gpt-5-nano',
    temperature=1
)

#### Load prompts templates

In [14]:
# Group prompts by category
prompts = {'monetary_stance': [], 'monetary_agreement': [], 'fiscal_stance': [], 'fiscal_agreement': []}
for key, entry in PROMPT_REGISTRY.items():
    fname = entry["prompt_file"]
    for category in prompts.keys():
        if fname.startswith(category):
            prompts[category].append(key)
            break

print(f"Loaded {sum(len(v) for v in prompts.values())} prompts across {len(prompts)} categories")

Loaded 16 prompts across 4 categories


In [10]:
prompts

{'monetary_stance': ['monetary_stance_simple',
  'monetary_stance_with_definitions',
  'monetary_stance_few_shot',
  'monetary_stance_chain_of_thought'],
 'monetary_agreement': ['monetary_agreement_simple',
  'monetary_agreement_with_definitions',
  'monetary_agreement_few_shot',
  'monetary_agreement_chain_of_thought'],
 'fiscal_stance': ['fiscal_stance_simple',
  'fiscal_stance_with_definitions',
  'fiscal_stance_few_shot',
  'fiscal_stance_chain_of_thought'],
 'fiscal_agreement': ['fiscal_agreement_simple',
  'fiscal_agreement_with_definitions',
  'fiscal_agreement_few_shot',
  'fiscal_agreement_chain_of_thought']}

## Load data

In [11]:
data_dir = Path(config.data_dir / 'Finetuning_data')
data_dir.exists()

True

In [12]:
df = pd.read_excel(data_dir / 'labelled_fiscal_v2.xlsx')
#df = pd.read_excel('/data/home/xiong/data/Fund/CSR/Tractions/Finetuning_data/labelled_fiscal_v2.xlsx')
df_agree = df[['index', 'Print ISBN', 'staff', 'buff', 'country', 'year', 'agreement_general', 'disagreement_areas']]
df_stance = pd.DataFrame()
for tp in ['staff', 'buff']:
    df_temp = df[['index', 'Print ISBN', 'country', 'year', tp, '%s_stance_near_term'%tp]]
    df_temp = df_temp.rename(columns={tp: 'text'}).rename(columns={c: c.replace(tp+'_', '') for c in df_temp.columns})
    df_temp['type'] = tp
    df_stance = pd.concat([df_stance, df_temp], ignore_index=True)
    
# Remove the 'index' column and reset index to create a new 'index' column
df_stance = df_stance.drop(columns=['index']).reset_index(names='index')

In [13]:
# prepare files for different tasks
# df_train = pd.read_excel( data_dir / 'Fiscal/cv/training_2.xlsx')
# df_test = pd.read_excel(data_dir / 'Fiscal/cv/testing_2.xlsx')

# df_train_agree = df_train[['index', 'Print ISBN', 'staff', 'buff', 'country', 'year', 'agreement_general', 'disagreement_areas']]
# df_test_agree = df_test[['index', 'Print ISBN', 'staff', 'buff', 'country', 'year', 'agreement_general', 'disagreement_areas']]

#### Work on agreement

In [7]:
for col in ['agreement_general', 'disagreement_areas']:
    print(df_agree[col].value_counts(normalize=True))

agreement_general
mostly agree           0.566667
disagreement exists    0.283333
irrelevant             0.150000
Name: proportion, dtype: float64
disagreement_areas
government debt & financing                                0.219780
government revenue                                         0.164835
economic fundamentals                                      0.131868
government expenditure                                     0.131868
near-term policy direction                                 0.120879
government debt & financing; near-term policy direction    0.054945
economic fundamentals; near-term policy direction          0.032967
fiscal framework                                           0.032967
government expenditure; near-term policy direction         0.021978
government revenue; near-term policy direction             0.021978
medium-term fiscal stance                                  0.021978
political cycle; economic fundamentals                     0.010989
medium-term fiscal

In [8]:
# Demo: Build batch messages for monetary agreement using flexible function
prompt_key = 'fiscal_agreement_few_shot'
registry_entry = PROMPT_REGISTRY[prompt_key]
prompt_file = registry_entry["prompt_file"]
prompt_dir = Path('../../src/Traction/prompts')
# Load prompt template
prompt_template = load_prompt(prompt_dir / prompt_file).sections
response_model = registry_entry["response_model"]

In [169]:
PROMPT_REGISTRY

{'topic_classification': {'prompt_file': 'topic_classification.md',
  'response_model': schema.TopicResponse},
 'monetary_stance_simple': {'prompt_file': 'monetary_stance_simple.md',
  'response_model': schema.MonetaryStanceResponse},
 'monetary_stance_with_definitions': {'prompt_file': 'monetary_stance_with_definitions.md',
  'response_model': schema.MonetaryStanceResponse},
 'monetary_stance_few_shot': {'prompt_file': 'monetary_stance_few_shot.md',
  'response_model': schema.MonetaryStanceResponse},
 'monetary_stance_chain_of_thought': {'prompt_file': 'monetary_stance_chain_of_thought.md',
  'response_model': schema.MonetaryStanceChainOfThoughtResponse},
 'monetary_agreement_simple': {'prompt_file': 'monetary_agreement_simple.md',
  'response_model': schema.MonetaryAgreementResponse},
 'monetary_agreement_with_definitions': {'prompt_file': 'monetary_agreement_with_definitions.md',
  'response_model': schema.MonetaryAgreementResponse},
 'monetary_agreement_few_shot': {'prompt_file': '

In [10]:
# Define column mapping: template placeholder -> dataframe column
# Template uses: {COUNTRY}, {YEAR}, {STAFF_TEXT}, {AUTHORITY_TEXT}
# DataFrame has: 'country', 'year', 'staff', 'buff'
column_mapping = {
    'COUNTRY': 'country',
    'YEAR': 'year',
    'STAFF_TEXT': 'staff',
    'AUTHORITY_TEXT': 'buff'
}

In [11]:
# combine both traing and testing data
# sample_df = pd.concat([df_train_agree, df_test_agree])
sample_df = df_agree

In [12]:
# Build batch messages from first 5 rows
batch_messages, batch_ids = _build_batch_messages_from_df_flexible(
    sample_df,
    prompt_template,
    column_mapping=column_mapping,
    id_column='index',
    max_text_length=10000
)

In [14]:
print(f"Generated {len(batch_messages)} batch messages")
print(f"Sample IDs: {batch_ids}")
print(f"\nFirst message structure:")
print(f"Number of message parts: {len(batch_messages[0])}")
for i, msg in enumerate(batch_messages[30][:4]):  # Show first 2 parts
    print(f"\nPart {i+1} - Role: {msg.get('role', 'N/A')}")
    content = msg.get('content', '')
    print(f"{content[:5500]}...")


Generated 300 batch messages
Sample IDs: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '90', '91', '92', '93', '94', '95', '96', '97', '98', '99', '100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '110', '111', '112', '113', '114', '115', '116', '117', '118', '119', '120', '121', '122', '123', '124', '125', '126', '127', '128', '129', '130', '131', '132', '133', '134', '135', '136', '137', '138', '139', '140', '141', '142', '143', '144', '145', '146', '147', '148', '149', '150', '151', '152

In [15]:
try:
    response = llm_agent.get_response_content(batch_messages[15], reasoning_effort='low', response_format=response_model)
    print(response)
except Exception as e:
    print(f"Error: {e}")

agreement='mostly agree' disagreement_areas='[]'


In [24]:
batch_agent = BatchAsyncLLMAgent(
    api_key=os.getenv('OPENAI_API_KEY'),
    model='gpt-4o-2024-08-06',
    temperature=0
)

In [25]:
# Process a small subset to demo
subset_messages = batch_messages
subset_ids = batch_ids

async def run_batch():
    results = await _process_batch_async(
        batch_agent,
        subset_messages,
        response_model=registry_entry["response_model"],
        batch_size=8,
        max_completion_tokens=5000,
        #reasoning_effort='low'
    )
    return results

results = asyncio.run(run_batch())

Processing batches: 100%|██████████| 300/300 [00:33<00:00,  9.03msg/s]


In [32]:
# Show merged outputs (id + parsed response)
merged = _merge_ids_with_responses(subset_ids, results)
print(f"Processed {len(merged)} responses")
print(merged[0])

Processed 300 responses
{'agreement': 'mostly agree', 'disagreement_areas': '[]', 'id': '0'}


In [33]:
# Convert merged results to DataFrame
df_merged = pd.DataFrame(merged)
# Add _llm suffix to column names from the LLM results
llm_columns = [col for col in df_merged.columns if col not in ['id']]
rename_dict = {col: f"{col}_llm" for col in llm_columns}
df_merged = df_merged.rename(columns=rename_dict)
df_merged

,agreement_llm,disagreement_areas_llm,id
0,mostly agree,[],0
1,mostly agree,[],1
2,disagreement exists,['economic fundamentals'],2
3,mostly agree,,3
4,mostly agree,,4
...,...,...,...
295,mostly agree,[],295
296,mostly agree,[],296
297,mostly agree,[],297
298,mostly agree,,298


In [34]:
# Fix the merge issue by converting id column types to match
df_merged['id'] = df_merged['id'].astype(int)
df_merged = df_merged.merge(sample_df, left_on='id', right_on='index', how='left')
# Keep only records with no NaN values in key columns
df_merged = df_merged.dropna(subset=['agreement_general', 'agreement_llm'])
print(len(df_merged))
df_merged.head()

300


,agreement_llm,disagreement_areas_llm,id,index,Print ISBN,staff,buff,country,year,agreement_general,disagreement_areas
0,mostly agree,[],0,0,9781513529943,55. Staff welcomes the authorities’ commitment...,Fiscal adjustment in 2014 was better than prog...,Tunisia,2015.0,mostly agree,NaN
1,mostly agree,[],1,1,9781484317259,52. In this encouraging yet fragile environmen...,Mindful of the implications of the large fisca...,Ghana,2017.0,mostly agree,NaN
2,disagreement exists,['economic fundamentals'],2,2,9781513506180,48. Credible and balanced fiscal consolidation...,5. Fiscal consolidation has made major progres...,Japan,2015.0,disagreement exists,economic fundamentals
3,mostly agree,,3,3,9781513579863,32. Saving the oil-duty windfall in 2015 would...,Fiscal policy continued to be prudent despite ...,Republic of Belarus,2015.0,mostly agree,NaN
4,mostly agree,,4,4,9781484309322,41. Execution of the FY16/17 budget points to ...,The medium term fiscal framework remains ancho...,Uganda,2017.0,mostly agree,NaN


In [35]:
# query v6:
print("accuracy Score:")
print(accuracy_score(df_merged['agreement_general'], 
               df_merged['agreement_llm']))

print("f1 Score:")      
print(f1_score(df_merged['agreement_general'], 
         df_merged['agreement_llm'], 
         average='weighted'))

print("Confusion Matrix:")
confusion_matrix(df_merged['agreement_general'], 
                 df_merged['agreement_llm'], 
                 labels=['disagreement exists', 
                         'mostly agree', 
                         'irrelevant'])

accuracy Score:
0.7166666666666667
f1 Score:
0.6615252167060678
Confusion Matrix:


array([[ 52,  33,   0],
       [  9, 161,   0],
       [  7,  36,   2]])

#### Try Stance prediction

In [60]:
# Demo: Build batch messages for monetary agreement using flexible function
prompt_key = 'fiscal_stance_chain_of_thought'
registry_entry = PROMPT_REGISTRY[prompt_key]
prompt_dir = Path('../../src/Traction/prompts')
prompt_file = registry_entry["prompt_file"]
prompt_template = load_prompt(prompt_dir / prompt_file).sections
response_model = registry_entry["response_model"]

In [61]:
prompt_template

{'system': 'You are an experienced macroeconomist from IMF. Given a piece of text concerning a particular country in a given year expressing the views of {TYPE}, classify the {TYPE_POSSESSIVE} {VERB} near-term (next year) direction of change in fiscal policy stance as described in the text into "tightening"/"tightening bias"/"no change"/"loosening bias"/"loosening"/"unclear"/"irrelevant". {EXPLANATION}\n\nDefinitions:\nTightening: Suggests a plan to reduce fiscal deficits or move towards a surplus. This can be achieved through higher taxation or non-tax revenues, reduced government spending, or a combination of both.\nTightening Bias: Indicates a leaning towards a tightening fiscal policy but without a full commitment.\nNo Change: Indicates a plan to maintain the current fiscal policy stance without significant changes to overall fiscal balance.\nLoosening Bias: Suggests a leaning towards adopting a loosening fiscal policy, though not explicitly planning to do so.\nLoosening: Suggests 

In [62]:
df_stance.head()

,index,Print ISBN,country,year,text,stance_near_term,type,examples,explanation,author_type,type_possessive,verb
0,0,9781513529943,Tunisia,2015.0,55. Staff welcomes the authorities’ commitment...,tightening,staff,Example 1: Country: Tunisia; Year: 2015; Text:...,Note that the near-term policy direction recom...,IMF staff,IMF staff's,recommended
1,1,9781484317259,Ghana,2017.0,52. In this encouraging yet fragile environmen...,tightening,staff,Example 1: Country: Tunisia; Year: 2015; Text:...,Note that the near-term policy direction recom...,IMF staff,IMF staff's,recommended
2,2,9781513506180,Japan,2015.0,48. Credible and balanced fiscal consolidation...,tightening,staff,Example 1: Country: Tunisia; Year: 2015; Text:...,Note that the near-term policy direction recom...,IMF staff,IMF staff's,recommended
3,3,9781513579863,Republic of Belarus,2015.0,32. Saving the oil-duty windfall in 2015 would...,tightening,staff,Example 1: Country: Tunisia; Year: 2015; Text:...,Note that the near-term policy direction recom...,IMF staff,IMF staff's,recommended
4,4,9781484309322,Uganda,2017.0,41. Execution of the FY16/17 budget points to ...,tightening bias,staff,Example 1: Country: Tunisia; Year: 2015; Text:...,Note that the near-term policy direction recom...,IMF staff,IMF staff's,recommended


In [63]:
for col in ['stance_near_term']:
    print(df_stance[col].value_counts(normalize=True))
    print('\n')

stance_near_term
tightening         0.546667
loosening          0.136667
unclear            0.111667
no change          0.088333
tightening bias    0.070000
loosening bias     0.046667
Name: proportion, dtype: float64




In [64]:
type_dict1 = {'staff': 'IMF staff', 'buff': 'a country\'s authorities'}
type_dict2 = {'staff': 'IMF staff\'s', 'buff': 'country\'s authorities\''}
verb_dict = {'staff': 'recommended', 'buff': 'planned'}
example_dict = {'staff': 'Example 1: Country: Tunisia; Year: 2015; Text: \"The modest fiscal loosening in 2015 appropriately responds to weaker economic activity. Going forward, fiscal consolidation is needed to reduce external imbalances, restore private sector confidence, and ensure debt sustainability.\" Answer: \"tightening\".\nExample 2: Country: Denmark; Year: 2019; Text: \"The fiscal stance should remain neutral, while letting automatic stabilizers operate fully in case of shocks to aggregate demand.\" Answer: \"no change\".\nExample 3: Country: Denmark; Year: 2017; Text: \"Thus, provided that strong new labor market reforms are agreed to raise labor supply, it would be appropriate to slow the pace of consolidation somewhat to facilitate cuts in the high tax burden.\" Answer: \"loosening bias\".',
               'buff': 'Example 1: Country: Tunisia; Year: 2015; Text: \"The authorities are firmly committed to take additional measures to attain their fiscal objectives in 2015, including through reduction of non-essential non-wage expenditure. They are committed to fiscal adjustment from 2016 onwards.\" Answer: \"tightening\".\nExample 2: Country: China; Year: 2019; Text: \"We concur with staff’s suggestion that there is no need for a further large-scale fiscal stimulus at this moment since the effects of trade tensions have already been factored into this year’s budget.\" Answer: \"no change\".\nExample 3: Country: Singapore; Year: 2019; Text: \"While fiscal policy is focused on medium- to long-term restructuring, the authorities stand ready to provide fiscal stimulus should economic conditions take a significant turn for the worse.\" Answer: \"loosening bias\".'}
explanation_dict = {'staff': 'Note that the near-term policy direction recommended by staff is different in concept from staff\'s projected near-term policy direction of the authorities.',
                    'buff': 'Note that the near-term policy direction planned by the authorities are different in concept from IMF staff\'s recommended or projected near-term policy direction.'}

# Add examples and explanation columns based on type
df_stance['examples'] = df_stance['type'].map(example_dict)
df_stance['explanation'] = df_stance['type'].map(explanation_dict)
df_stance['author_type'] = df_stance['type'].map(type_dict1)
df_stance['type_possessive'] = df_stance['type'].map(type_dict2)
df_stance['verb'] = df_stance['type'].map(verb_dict)


In [65]:
# Define column mapping: template placeholder -> dataframe column
# Template uses: {COUNTRY}, {YEAR}, {STAFF_TEXT}, {AUTHORITY_TEXT}
# DataFrame has: 'country', 'year', 'staff', 'buff'
column_mapping = {
    'COUNTRY': 'country',
    'YEAR': 'year',
    'TEXT': 'text',
    'TYPE': 'author_type',
    'TYPE_POSSESSIVE': 'type_possessive',
    'VERB': 'verb',
    'EXAMPLES': 'examples',
    'EXPLANATION': 'explanation'
}

In [66]:
# Build batch messages from first 5 rows
batch_messages, batch_ids = _build_batch_messages_from_df_flexible(
    df_stance,
    prompt_template,
    column_mapping=column_mapping,
    id_column='index',
    max_text_length=8000
)

In [67]:
print(f"Generated {len(batch_messages)} batch messages")
print(f"Sample IDs: {batch_ids}")
print(f"\nFirst message structure:")
print(f"Number of message parts: {len(batch_messages[0])}")
for i, msg in enumerate(batch_messages[30][:4]):  # Show first 2 parts
    print(f"\nPart {i+1} - Role: {msg.get('role', 'N/A')}")
    content = msg.get('content', '')
    print(f"{content[:3000]}...")
""

Generated 600 batch messages
Sample IDs: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '90', '91', '92', '93', '94', '95', '96', '97', '98', '99', '100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '110', '111', '112', '113', '114', '115', '116', '117', '118', '119', '120', '121', '122', '123', '124', '125', '126', '127', '128', '129', '130', '131', '132', '133', '134', '135', '136', '137', '138', '139', '140', '141', '142', '143', '144', '145', '146', '147', '148', '149', '150', '151', '152

''

In [81]:
batch_agent = BatchAsyncLLMAgent(
    api_key=os.getenv('OPENAI_API_KEY'),
    model='gpt-5-mini',#'gpt-4o-2024-08-06',
    temperature=1
)

In [82]:
# Process a small subset to demo
subset_messages = batch_messages
subset_ids = batch_ids

async def run_batch():
    results = await _process_batch_async(
        batch_agent,
        subset_messages,
        response_model=registry_entry["response_model"],
        batch_size=8,
        max_completion_tokens=5000,
        #reasoning_effort='low'
    )
    return results

results = asyncio.run(run_batch())

Processing batches: 100%|██████████| 600/600 [15:25<00:00,  1.54s/msg]


In [83]:
# Show merged outputs (id + parsed response)
merged = _merge_ids_with_responses(subset_ids, results)
print(f"Processed {len(merged)} responses")

# Convert merged results to DataFrame
df_merged = pd.DataFrame(merged)
# Add _llm suffix to column names from the LLM results
llm_columns = [col for col in df_merged.columns if col not in ['id']]
rename_dict = {col: f"{col}_llm" for col in llm_columns}
df_merged = df_merged.rename(columns=rename_dict)
df_merged

Processed 600 responses


,reason_llm,stance_near_term_llm,id
0,Staff notes a modest fiscal loosening in 2015 ...,tightening,0
1,Staff explicitly calls for credible and sustai...,tightening,1
2,The staff calls explicitly for “credible and b...,tightening,2
3,IMF staff explicitly recommends saving the 201...,tightening,3
4,Staff recommends a very tight current expendit...,tightening,4
...,...,...,...
595,The authorities repeatedly state commitment to...,tightening,595
596,The authorities state that fiscal consolidatio...,tightening,596
597,Although authorities maintain emergency suppor...,tightening,597
598,Authorities state the five-year budget “envisa...,tightening,598


In [84]:
# Fix the merge issue by converting id column types to match
df_merged['id'] = df_merged['id'].astype(int)
df_merged = df_merged.merge(df_stance, left_on='id', right_on='index', how='left')
# Keep only records with no NaN values in key columns
df_merged = df_merged.dropna(subset=[ 'stance_near_term'])
print(len(df_merged))
df_merged.head()

600


,reason_llm,stance_near_term_llm,id,index,Print ISBN,country,year,text,stance_near_term,type,examples,explanation,author_type,type_possessive,verb
0,Staff notes a modest fiscal loosening in 2015 ...,tightening,0,0,9781513529943,Tunisia,2015.0,55. Staff welcomes the authorities’ commitment...,tightening,staff,Example 1: Country: Tunisia; Year: 2015; Text:...,Note that the near-term policy direction recom...,IMF staff,IMF staff's,recommended
1,Staff explicitly calls for credible and sustai...,tightening,1,1,9781484317259,Ghana,2017.0,52. In this encouraging yet fragile environmen...,tightening,staff,Example 1: Country: Tunisia; Year: 2015; Text:...,Note that the near-term policy direction recom...,IMF staff,IMF staff's,recommended
2,The staff calls explicitly for “credible and b...,tightening,2,2,9781513506180,Japan,2015.0,48. Credible and balanced fiscal consolidation...,tightening,staff,Example 1: Country: Tunisia; Year: 2015; Text:...,Note that the near-term policy direction recom...,IMF staff,IMF staff's,recommended
3,IMF staff explicitly recommends saving the 201...,tightening,3,3,9781513579863,Republic of Belarus,2015.0,32. Saving the oil-duty windfall in 2015 would...,tightening,staff,Example 1: Country: Tunisia; Year: 2015; Text:...,Note that the near-term policy direction recom...,IMF staff,IMF staff's,recommended
4,Staff recommends a very tight current expendit...,tightening,4,4,9781484309322,Uganda,2017.0,41. Execution of the FY16/17 budget points to ...,tightening bias,staff,Example 1: Country: Tunisia; Year: 2015; Text:...,Note that the near-term policy direction recom...,IMF staff,IMF staff's,recommended


In [85]:
df_merged['stance_near_term_llm'].value_counts()

stance_near_term_llm
tightening         377
loosening           92
tightening bias     61
no change           42
loosening bias      21
unclear              7
Name: count, dtype: int64

In [86]:
df_merged['stance_near_term'].value_counts()

stance_near_term
tightening         328
loosening           82
unclear             67
no change           53
tightening bias     42
loosening bias      28
Name: count, dtype: int64

In [87]:
print('raw:', accuracy_score(df_merged['stance_near_term'], df_merged['stance_near_term_llm']), f1_score(df_merged['stance_near_term'], df_merged['stance_near_term_llm'], average='weighted'), 'merging unclear/irrelevant:', \
accuracy_score(df_merged['stance_near_term'].apply(lambda x: 'unclear' if x=='irrelevant' else x), df_merged['stance_near_term_llm'].apply(lambda x: 'unclear' if x=='irrelevant' else x)), f1_score(df_merged['stance_near_term'].apply(lambda x: 'unclear' if x=='irrelevant' else x), df_merged['stance_near_term_llm'].apply(lambda x: 'unclear' if x=='irrelevant' else x), average='weighted'))


raw: 0.6616666666666666 0.6337617399879677 merging unclear/irrelevant: 0.6616666666666666 0.6337617399879677


In [88]:
md='llm'
accuracy_score(df_merged['stance_near_term'].apply(lambda x: 'unclear' if x=='irrelevant' else x.replace(' bias', '')), df_merged['stance_near_term_%s'%md].apply(lambda x: 'unclear' if x=='irrelevant' else x.replace(' bias', ''))), f1_score(df_merged['stance_near_term'].apply(lambda x: 'unclear' if x=='irrelevant' else x.replace(' bias', '')), df_merged['stance_near_term_%s'%md].apply(lambda x: 'unclear' if x=='irrelevant' else x.replace(' bias', '')), average='weighted')


(0.7816666666666666, 0.7427494973001308)

In [89]:
# query v1:
print("Stance Future - Accuracy:", 
      accuracy_score(df_merged['stance_near_term'], df_merged['stance_near_term_llm']))
print("Stance Future - F1 Score:", 
      f1_score(df_merged['stance_near_term'], df_merged['stance_near_term_llm'], average='weighted'))

print("\nMerging unclear/irrelevant and bias categories:")
print("Stance Future - Accuracy:", 
      accuracy_score(df_merged['stance_near_term'].apply(lambda x: 'unclear' if x=='irrelevant' else x.replace(' bias', '')), 
                     df_merged['stance_near_term_llm'].apply(lambda x: 'unclear' if x=='irrelevant' else x.replace(' bias', ''))))
print("Stance Future - F1 Score:", 
      f1_score(df_merged['stance_near_term'].apply(lambda x: 'unclear' if x=='irrelevant' else x.replace(' bias', '')), 
               df_merged['stance_near_term_llm'].apply(lambda x: 'unclear' if x=='irrelevant' else x.replace(' bias', '')), average='weighted'))

Stance Future - Accuracy: 0.6616666666666666
Stance Future - F1 Score: 0.6337617399879677

Merging unclear/irrelevant and bias categories:
Stance Future - Accuracy: 0.7816666666666666
Stance Future - F1 Score: 0.7427494973001308
